### Structured Output

Models can be requested to provide their response in a format matching a given schema. this is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for ensuring structured data.

### Pydantic Models
Pydantic models provide the richest feature set with field validation, descriptions and nested structure.

In [1]:
import os

from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-safeguard-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000270465E1FD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000270465E2CF0>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000270465E1FD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000270465E2CF0>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the m

In [4]:
model.invoke("Provide details about the movie Inception")

AIMessage(content="**Inception (2010)** – A science‑fiction psychological thriller directed by Christopher\u202fNolan.\n\n| Item | Details |\n|------|---------|\n| **Release date** | 16\u202fJuly\u202f2010 (US) |\n| **Runtime** | 2\u202fh\u202f28\u202fmin |\n| **Country** | United States / United Kingdom |\n| **Language** | English |\n| **Budget** | ≈\u202f160\u202fmillion\u202fUSD |\n| **Box‑office** | ≈\u202f829\u202fmillion\u202fUSD worldwide |\n| **Production companies** | Warner\u202fBros. Pictures, Syncopy Inc. |\n| **Screenplay** | Christopher\u202fNolan |\n| **Director** | Christopher\u202fNolan |\n| **Music** | Hans\u202fZimmer |\n| **Cinematography** | Wally\u202fPfister |\n| **Editor** | Lee\u202fSmith |\n| **Studio** | Syncopy |\n\n---\n\n## 1. Premise\n\n**Inception** follows Dom\u202fCobb (Leonardo\u202fDiCaprio), a “extractor” who enters others’ dreams to steal or plant ideas. After a failed job, he is offered a chance to regain his family by performing the reverse – *in

In [5]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title:str = Field(..., description="The title of the movie")
    year: int = Field(... , description="This year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

In [7]:
response = model_with_structure.invoke("Provide details about movie Incetption")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide details about movie Incetption". Likely they want details about "Inception". There\'s a typo. They want movie details. We can call the function to get details. Let\'s use the function "Movie" with title "Inception". Also provide director, rating, year. We\'ll call it.', 'tool_calls': [{'id': 'fc_4be41525-9d7b-4ee8-a953-e2ef3f6749ed', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 163, 'total_tokens': 269, 'completion_time': 0.109287178, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.013647257, 'prompt_tokens_details': None, 'queue_time': 0.260634391, 'total_time': 0.122934435}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'f

### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [11]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ... , "This year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]


model_with_typedDict = model.with_structured_output(MovieDict)
response = model_with_typedDict.invoke("Provide details about the movie Inception")
response

{'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [12]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [13]:
model.profile

{'name': 'Safety GPT OSS 20B',
 'release_date': '2025-03-05',
 'last_updated': '2025-03-05',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A Data class is a class typically containing mainly data, although there aren't really any restrictions. you create it using the @dataclass decorator.

In [1]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [2]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-safeguard-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='c70672d4-74b1-4b33-9d3c-882ef4fed7ab'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Extract contact info from: John Doe, john@example.com, (555) 123-4567". We need to call the tool functions.ContactInfo with name, email, phone. Provide JSON? The function expects email, name, phone. We need to produce the function call. The assistant should use the tool. So produce a function call.', 'tool_calls': [{'id': 'fc_53e46475-12e5-4fc2-b20b-f5acd5f65ba7', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 167, 'total_tokens': 284, 'completion_time': 0.119701017, 'completion_tokens_details': {'reasoning_tokens': 76}, 'prompt_time': 0.

In [3]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [5]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-safeguard-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [7]:
### Dataclass


from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact information for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model="groq:openai/gpt-oss-safeguard-20b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']



ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')